# 7. Balanced assignment, written with sets and dictionaries

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 20–23. The third version: the
workers and jobs are numbered 1–4 as in the table, the times are in a dictionary keyed by (worker, job), and
Pyomo builds the constraints from two rules.

**The problem.** Four workers must do four jobs, one job each. The table gives the time each worker needs for
each job (page 20):

| | Job 1 | Job 2 | Job 3 | Job 4 |
| --- | --- | --- | --- | --- |
| Worker 1 | 9 | 2 | 7 | 8 |
| Worker 2 | 6 | 4 | 3 | 7 |
| Worker 3 | 5 | 8 | 1 | 8 |
| Worker 4 | 7 | 6 | 9 | 4 |

Minimise the total time. $x_{uv} = 1$ if worker $u$ does job $v$ (page 22):

minimise $\sum_{u \in W} \sum_{v \in J} c_{uv} x_{uv}$ subject to $\sum_{v \in J} x_{uv} = 1$ for every worker $u$,
$\sum_{u \in W} x_{uv} = 1$ for every job $v$, $x_{uv} \in \{0, 1\}$.

"Pick one cell in every row and every column": a perfect matching in the bipartite graph of workers and jobs.
There are $4! = 24$ possible assignments.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Total time **13**: worker 1 → job 2, worker 2 → job 1, worker 3 → job 3, worker 4 → job 4
(`x[1,2]`, `x[2,1]`, `x[3,3]`, `x[4,4]` = 1), the same as demos 5 and 6.

*Original script: `5 Source Files/Python/balanced_assignment_sets_dictionaries.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model, the two sets (numbered 1–4 here), one binary variable per (worker, job) pair, and the table of times as a dictionary.

In [ ]:
# Create the model, the sets and the binary variables: x[w, j] = 1 if worker w does job j
model = ConcreteModel()

workers = [1,2,3,4]
jobs = [1,2,3,4]

model.x = Var(workers, jobs, within=Binary)

job_time = {   # (worker, job): time, from the table on page 20
    (1,1): 9, (1,2):2, (1,3): 7, (1,4):8, 
    (2,1): 6, (2,2):4, (2,3): 3, (2,4):7, 
    (3,1): 5, (3,2):8, (3,3): 1, (3,4):8, 
    (4,1): 7, (4,2):6, (4,3): 9, (4,4):4
    }

The objective, then two rules: Pyomo calls `worker_constraint` once for each worker and `job_constraint` once for each job.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(sum(job_time[w, j] * model.x[w, j] for w in workers) for j in jobs), sense=minimize)

# Each worker does exactly one job
def worker_constraint(model, w):
    return sum(model.x[w, j] for j in jobs) == 1

# Each job gets exactly one worker
def job_constraint(model, j):
    return sum(model.x[w, j] for w in workers) == 1

model.worker_constraint = Constraint(workers, rule=worker_constraint)
model.job_constraint = Constraint(jobs, rule=job_constraint)

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: one constraint per worker and one per job.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the assignment (variables with value 1) and the total time.

In [ ]:
# Display the results
model.display()

## Try it

A fifth worker and a fifth job are added. What has to change in this notebook, and what does not?

<details><summary>Answer</summary>

Add `5` to `workers` and to `jobs`, and add the nine new times to `job_time` (worker 5 with jobs 1–5, and
workers 1–4 with job 5). The objective and the two rules do not change. With 5 workers there are
$5! = 120$ possible assignments; the solver still checks them without listing them one by one.
</details>